# 🤖 Notebook 5 — Model Building
**Goal:** Train 5 ML models (including XGBoost), compare performance, and save the best model automatically.

---

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import joblib
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/featured_data.csv')
print('Shape:', df.shape)
df.head()


Shape: (70100, 34)


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,...,Seasonality_enc,Sales_Lag1,Sales_Lag7,Sales_Lag14,Sales_Lag30,Rolling_7day_avg,Rolling_14day_avg,Rolling_30day_avg,Store_Cat,Store_Cat_enc
0,2022-01-31,S001,P0001,Clothing,West,410,200,152,212.24,70.78,...,2,130.0,9.0,69.0,127.0,97.857143,130.500000,114.233333,S001_Clothing,0
1,2022-02-01,S001,P0001,Groceries,East,419,279,84,297.26,34.49,...,2,200.0,189.0,123.0,81.0,125.142857,139.857143,116.666667,S001_Groceries,3
2,2022-02-02,S001,P0001,Electronics,South,415,38,149,53.13,52.49,...,0,279.0,49.0,211.0,5.0,138.000000,151.000000,123.266667,S001_Electronics,1
3,2022-02-03,S001,P0001,Clothing,South,345,71,186,84.02,27.71,...,3,38.0,164.0,79.0,58.0,136.428571,138.642857,124.366667,S001_Clothing,0
4,2022-02-04,S001,P0001,Furniture,South,121,25,25,37.72,16.84,...,0,71.0,97.0,390.0,147.0,123.142857,138.071429,124.800000,S001_Furniture,2


## Define Features and Target

In [2]:
features = [
    'Month','Week','DayOfWeek','Year','Quarter','IsWeekend',
    'Price','Discount','Competitor Pricing','Inventory Level',
    'Units Ordered','Holiday/Promotion',
    'Category_enc','Region_enc','Weather Condition_enc','Seasonality_enc',
    'Sales_Lag1','Sales_Lag7','Sales_Lag14','Sales_Lag30',
    'Rolling_7day_avg','Rolling_14day_avg','Rolling_30day_avg',
    'Store_Cat_enc'
]

# Only keep features that exist in the dataframe
features = [f for f in features if f in df.columns]

X = df[features]
y = df['Units Sold']

print('Features used:', len(features))
print(features)


Features used: 23
['Month', 'Week', 'DayOfWeek', 'Year', 'Quarter', 'IsWeekend', 'Price', 'Discount', 'Competitor Pricing', 'Inventory Level', 'Units Ordered', 'Holiday/Promotion', 'Category_enc', 'Region_enc', 'Seasonality_enc', 'Sales_Lag1', 'Sales_Lag7', 'Sales_Lag14', 'Sales_Lag30', 'Rolling_7day_avg', 'Rolling_14day_avg', 'Rolling_30day_avg', 'Store_Cat_enc']


## Train/Test Split
**What:** We keep 80% of data for training and 20% for testing (evaluating model performance on unseen data).

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Training samples : {X_train.shape[0]:,}')
print(f'Test samples     : {X_test.shape[0]:,}')


Training samples : 56,080
Test samples     : 14,020


## Helper Function to Evaluate Models

In [4]:
results = []

def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    preds       = model.predict(X_test)
    train_preds = model.predict(X_train)
    mae   = mean_absolute_error(y_test, preds)
    rmse  = np.sqrt(mean_squared_error(y_test, preds))
    r2    = r2_score(y_test, preds)
    r2_tr = r2_score(y_train, train_preds)
    print(f'\n--- {name} ---')
    print(f'  MAE         : {mae:.2f}')
    print(f'  RMSE        : {rmse:.2f}')
    print(f'  R² (test)   : {r2:.4f}')
    print(f'  R² (train)  : {r2_tr:.4f}')
    overfit = r2_tr - r2
    if overfit > 0.3:
        print(f'  Overfit     : ⚠️  Heavy ({overfit:.2f} gap)')
    elif overfit > 0.05:
        print(f'  Overfit     : ⚡ Slight ({overfit:.2f} gap)')
    else:
        print(f'  Overfit     : ✅ None ({overfit:.2f} gap)')
    results.append({'model': name, 'obj': model, 'mae': mae, 'rmse': rmse, 'r2': r2})
    return model, preds


## Model 1 — Linear Regression
**What:** Finds the best straight line through the data. Simple, fast, and surprisingly effective.

In [5]:
lr_model, lr_preds = evaluate_model(
    'Linear Regression', LinearRegression(), X_train, X_test, y_train, y_test
)



--- Linear Regression ---
  MAE         : 68.90
  RMSE        : 87.88
  R² (test)   : 0.3456
  R² (train)  : 0.3491
  Overfit     : ✅ None (0.00 gap)


## Model 2 — Decision Tree
**What:** Splits data into branches like a flowchart (if price > 50 → branch A, else → branch B).

In [6]:
dt_model, dt_preds = evaluate_model(
    'Decision Tree', DecisionTreeRegressor(max_depth=8, random_state=42), X_train, X_test, y_train, y_test
)



--- Decision Tree ---
  MAE         : 69.61
  RMSE        : 89.03
  R² (test)   : 0.3284
  R² (train)  : 0.3678
  Overfit     : ✅ None (0.04 gap)


## Model 3 — Random Forest
**What:** Builds 100 decision trees and averages their predictions. Usually very accurate but can overfit.

In [7]:
rf_model, rf_preds = evaluate_model(
    'Random Forest', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1), X_train, X_test, y_train, y_test
)



--- Random Forest ---
  MAE         : 69.49
  RMSE        : 88.86
  R² (test)   : 0.3309
  R² (train)  : 0.9067
  Overfit     : ⚠️  Heavy (0.58 gap)


## Model 4 — Gradient Boosting
**What:** Builds trees one by one, each one correcting the errors of the previous one.

In [8]:
gb_model, gb_preds = evaluate_model(
    'Gradient Boosting', GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42), X_train, X_test, y_train, y_test
)



--- Gradient Boosting ---
  MAE         : 69.00
  RMSE        : 87.95
  R² (test)   : 0.3446
  R² (train)  : 0.3585
  Overfit     : ✅ None (0.01 gap)


## Model 5 — XGBoost ⭐ NEW
**What:** An optimized, faster version of Gradient Boosting. Handles missing values natively and often achieves the best scores.

In [8]:
xgb_model, xgb_preds = evaluate_model(
    'XGBoost',
    xgb.XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbosity=0
    ),
    X_train, X_test, y_train, y_test
)



--- XGBoost ---
  MAE         : 69.12
  RMSE        : 88.12
  R² (test)   : 0.3421
  R² (train)  : 0.4330
  Overfit     : ⚡ Slight (0.09 gap)


## Compare All Models & Save the Best One Automatically

In [9]:
# Build comparison table
results_df = pd.DataFrame([
    {'Model': r['model'], 'MAE': round(r['mae'],2), 'RMSE': round(r['rmse'],2), 'R²': round(r['r2'],4)}
    for r in results
])
results_df = results_df.sort_values('R²', ascending=False).reset_index(drop=True)
results_df.index += 1
print('\n=== MODEL COMPARISON ===')
print(results_df.to_string())

# Auto-pick the best model by R² score
best = max(results, key=lambda x: x['r2'])
print(f'\n🏆 Best Model: {best["model"]}  (R² = {best["r2"]:.4f})')

# Save the best model
joblib.dump(best['obj'], '../models/best_model.pkl')
joblib.dump(features,   '../models/feature_list.pkl')

# Also keep linear_regression_model.pkl so the Flask app still works
for r in results:
    if r['model'] == 'Linear Regression':
        joblib.dump(r['obj'], '../models/linear_regression_model.pkl')

print(f'\n✅ Best model saved  → ../models/best_model.pkl')
print(f'✅ Feature list saved → ../models/feature_list.pkl')
print(f'✅ Linear model saved → ../models/linear_regression_model.pkl (Flask fallback)')



=== MODEL COMPARISON ===
               Model    MAE   RMSE      R²
1  Linear Regression  68.90  87.88  0.3456
2            XGBoost  69.12  88.12  0.3421
3            XGBoost  69.12  88.12  0.3421
4      Decision Tree  69.61  89.03  0.3284

🏆 Best Model: Linear Regression  (R² = 0.3456)

✅ Best model saved  → ../models/best_model.pkl
✅ Feature list saved → ../models/feature_list.pkl
✅ Linear model saved → ../models/linear_regression_model.pkl (Flask fallback)
